# Unsupervised Learning with 1000-Instance Car Sales Dataset

This notebook uses the `CarSalesPreprocessingDataset.csv` dataset, which contains exactly 1000 rows and 5 original features. We will implement **K-Means** and **DBSCAN**, compare them using clustering metrics, and evaluate proxy accuracy using a derived price category.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    accuracy_score,
)

sns.set(style="whitegrid")

## Load and Explore Dataset

Load the 1000-instance dataset and inspect its shape, feature names, and first rows.

In [ ]:
path = "CarSalesPreprocessingDataset.csv"
df = pd.read_csv(path)

print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
print("Only 1000 rows and 5 original features are present.")

df.head()

In [ ]:
# Summary statistics for numeric columns
display(df.describe(include=[np.number]).T)

In [ ]:
# Show distribution of the categorical features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Make", order=df["Make"].value_counts().index, ax=axes[0])
axes[0].set_title("Make distribution")
sns.countplot(data=df, x="Colour", order=df["Colour"].value_counts().index, ax=axes[1])
axes[1].set_title("Colour distribution")
plt.tight_layout()
plt.show()

## Preprocess Data

Convert the price field to numeric, encode categorical features, and scale the data.

A derived binary price category is also created for accuracy-style evaluation.

In [ ]:
# Convert price to numeric
price_numeric = df["Price"].str.replace("Rs", "", regex=False).astype(float)
df["Price_numeric"] = price_numeric

# Create a binary price category for proxy evaluation
price_median = df["Price_numeric"].median()
df["price_category"] = (df["Price_numeric"] > price_median).astype(int)

# Prepare feature matrix with original 5 fields, encoding categoricals
X = df[["Make", "Colour", "Odometer (KM)", "Doors", "Price_numeric"]]
X_encoded = pd.get_dummies(X, columns=["Make", "Colour"], drop_first=True)

y = df["price_category"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print("Encoded feature shape:", X_encoded.shape)
print("Scaled feature shape:", X_scaled.shape)

In [ ]:
# Visualize the dataset with PCA projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="coolwarm", s=15, alpha=0.7)
plt.title("PCA projection of car sales data (by derived price category)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Implement K-Means Clustering

Apply K-Means clustering with 2 clusters, since the derived price category is binary.
Use the elbow method to confirm the cluster choice.

In [ ]:
# Elbow method for KMeans
inertia = []
K = range(2, 7)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inertia, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.xticks(K)
plt.show()

In [ ]:
# KMeans clustering with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap="viridis", s=15)
plt.title("K-Means clusters on PCA projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Implement DBSCAN Clustering

Apply DBSCAN and tune `eps` and `min_samples` to find a cluster structure in the car sales data.

In [ ]:
# Evaluate candidate DBSCAN parameters
candidates = [0.5, 0.7, 0.9, 1.1]
results_dbscan = []
for eps in candidates:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_scaled)
    results_dbscan.append((eps, len(np.unique(labels)), np.sum(labels == -1)))

pd.DataFrame(results_dbscan, columns=["eps", "num_clusters_or_noise", "noise_count"])

In [ ]:
# Selected DBSCAN setting
best_eps = 0.9
dbscan = DBSCAN(eps=best_eps, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels, cmap="plasma", s=15)
plt.title(f"DBSCAN clusters on PCA projection (eps={best_eps})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Evaluate Clustering Performance

Use silhouette score, Calinski-Harabasz index, Davies-Bouldin index, and proxy accuracy against the derived price category.

In [ ]:
from itertools import permutations


def best_cluster_accuracy(true_labels, cluster_labels):
    valid_idx = cluster_labels != -1
    if np.sum(valid_idx) == 0:
        return np.nan
    clusters = np.unique(cluster_labels[valid_idx])
    labels = np.unique(true_labels[valid_idx])
    if len(clusters) != len(labels):
        return np.nan
    best_acc = 0.0
    for perm in permutations(labels):
        mapped = np.copy(cluster_labels[valid_idx])
        for cluster, label in zip(clusters, perm):
            mapped[cluster_labels[valid_idx] == cluster] = label
        best_acc = max(best_acc, accuracy_score(true_labels[valid_idx], mapped))
    return best_acc

kmeans_metrics = {
    "silhouette": silhouette_score(X_scaled, kmeans_labels),
    "calinski_harabasz": calinski_harabasz_score(X_scaled, kmeans_labels),
    "davies_bouldin": davies_bouldin_score(X_scaled, kmeans_labels),
    "accuracy": best_cluster_accuracy(y, kmeans_labels),
}

mask = dbscan_labels != -1
if np.unique(dbscan_labels[mask]).size > 1:
    dbscan_metrics = {
        "silhouette": silhouette_score(X_scaled[mask], dbscan_labels[mask]),
        "calinski_harabasz": calinski_harabasz_score(X_scaled[mask], dbscan_labels[mask]),
        "davies_bouldin": davies_bouldin_score(X_scaled[mask], dbscan_labels[mask]),
        "accuracy": best_cluster_accuracy(y[mask], dbscan_labels[mask]),
        "noise_points": np.sum(~mask),
    }
else:
    dbscan_metrics = {
        "silhouette": np.nan,
        "calinski_harabasz": np.nan,
        "davies_bouldin": np.nan,
        "accuracy": np.nan,
        "noise_points": np.sum(~mask),
    }

metrics_df = pd.DataFrame([kmeans_metrics, dbscan_metrics], index=["KMeans", "DBSCAN"])
metrics_df

## Compare Algorithms

Compare the clustering metrics and determine which algorithm performs better for this dataset.

In [ ]:
print("K-Means metrics:\n", metrics_df.loc["KMeans"])
print("\nDBSCAN metrics:\n", metrics_df.loc["DBSCAN"])

best_name = "KMeans" if metrics_df.loc["KMeans", "silhouette"] > metrics_df.loc["DBSCAN", "silhouette"] else "DBSCAN"
print(f"\nBased on silhouette score, the better algorithm is: {best_name}")

if metrics_df.loc["KMeans", "accuracy"] > metrics_df.loc["DBSCAN", "accuracy"]:
    print("K-Means also has higher proxy accuracy vs. the derived price category.")
else:
    print("DBSCAN has higher proxy accuracy or comparable performance.")

## Conclusion

- Dataset used: `CarSalesPreprocessingDataset.csv` with exactly 1000 rows and 5 original features.
- Two unsupervised algorithms were implemented: **K-Means** and **DBSCAN**.
- Proxy accuracy was evaluated against a derived binary price category, while clustering quality was measured using silhouette, Calinski-Harabasz, and Davies-Bouldin scores.
- The best algorithm is the one with the strongest silhouette score and proxy accuracy on this dataset.
